# ZenteiQ AI Tech Innovations — Task 3: MoE model (DeepSeek)

This notebook provides a detailed setup and execution workflow for training the **DeepSeek-16B-MOE** base model on a **GPU-only** environment using **MaxText** and **JAX** with synthetic data.

### 1. Installation & Environment Setup

In [1]:
# Clone the MaxText repository
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# Install uv (fast package installer)
!pip install uv

# Install MaxText GPU dependencies
!uv pip install -e .[cuda12]

# Restart the runtime after this step if run in Colab!

Cloning into 'maxtext'...
remote: Enumerating objects: 97136, done.
remote: Counting objects: 100% (2314/2314), done.
remote: Compressing objects: 100% (1174/1174), done.
remote: Total 97136 (delta 1885), reused 1157 (delta 1140), pack-reused 94822 (from 5)
Receiving objects: 100% (97136/97136), 411.71 MiB | 20.18 MiB/s, done.
Resolving deltas: 100% (70829/70829), done.
/content/maxtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 14.2 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 252 packages in 2.72s                                       
Prepared 116 packages in 4m 49s                                          
Uninstalled 54 packages in 602ms
Installed 117 packages in 504msax==2.16.0                   
 - absl-py==1.4.0
 + absl-py==2.4.0
 - aiofiles==24.1.0
 + aiofiles==25.1.0
 + aqtp==0.9.0
 + astroid==4.0.4
 + auditwheel==6.7.0
 + black==25.12.0
 + build==1.5.0
 + cfgv==3.5.0
 + chex==0.1.92
 + cloud-accelerator-diagnostics==0.1.1
 + cl

In [2]:
!uv pip install qwix tokamax -q

In [3]:
!uv pip install aqtp -q

### 2. Verify JAX GPU Backend Connection

Run the following cell to confirm that JAX successfully detects the GPU backend.

In [14]:
import jax
print("JAX Version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX Version: 0.10.2
Available devices: [CudaDevice(id=0)]
Default backend: gpu


### 3. Configuration Breakdown

Below are the parameters we are setting for this training run, along with their purpose and effect:

| Configuration Parameter | Value | Purpose | Effect |
| :--- | :--- | :--- | :--- |
| **`model_name`** | `deepseek (scaled-down MoE)` | Specifies the base DeepSeek Mixture-of-Experts (MoE) architecture to use. | Loads the DeepSeek model configuration and initializes the transformer with MoE-specific components such as expert layers and routing mechanisms. |
| **`steps`** | `50` | Defines training duration. | Runs the optimization/training loop for exactly 50 iterations, which is sufficient to compile the graph, calculate device step times, and measure TFLOPs throughput. |
| **`dataset_type`** | `synthetic` | Determines the data loader pipeline format. | Bypasses external dataset downloading and disk I/O bottlenecks by dynamically generating random tensors of input tokens on GPU memory. This allows pure computational throughput benchmarking. |
| **`base_output_directory`** | `./output-deepseek-gpu` | Sets the file storage root path. | All metrics, TensorBoard execution logs, compile metadata, and model checkpoints are saved under this root directory. |
| **`run_name`** | `deepseek-gpu` | Uniquely names the specific execution run. | Prevents overwriting other runs and creates nested folders named `deepseek-gpu/` for output checkpoints and statistics. |

### 4. Execute GPU Training (Deepseel MOE 16B)

We run the main training script with our configuration overrides. We capture the stdout logs to parse metrics later.

In [10]:
!sudo apt-get update && sudo apt-get install -y \
    libxcomposite1 \
    libxcursor1 \
    libxdamage1 \
    libxext6 \
    libxfixes3 \
    libxi6 \
    libxrandr2 \
    libxrender1 \
    libxtst6 \
    libgbm1 \
    libasound2



Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:7 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]


In [26]:
# Run training for 50 steps on GPU
!python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
    hardware=gpu \
    model_name=deepseek2-16b\
    steps=50 \
    dataset_type=synthetic \
    base_output_directory=$(pwd)/gpu-output-deepseek2-16b \
    run_name=deepseek2-16b-gpu\
    megablox=false \
    override_model_config=true \
    skip_jax_distributed_system=true \
    base_emb_dim=1152 \
    base_mlp_dim=5120 \
    base_moe_mlp_dim=768 \
    base_num_decoder_layers=12 \
    num_experts=16 \
    num_experts_per_tok=2 \
    shared_experts=1 \
    per_device_batch_size=1 \
    attention=dot_product \
    max_target_length=1024 \
    weight_dtype=bfloat16 \
    enable_checkpointing=false

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
I0618 17:57:14.082544 134057933079680 max_utils.py:245] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
I0618 17:57:14.172140 134057933079680 xla_bridge.py:849] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0618 17:57:14.404694 134057933079680 pyconfig.py:559] Config param abort_on_inf_loss: True
I0618 17:57:14.404824 134057933079680 pyconfig.py:559] Config param abort_on_

In [6]:
!uv pip install aqtp ml-goodput-measurement ml-collections -q

In [7]:
!uv pip install pathwaysutils aqt -q

In [8]:
!uv pip install drjax -q

In [12]:
import os
import sys

# Detect Python's nvidia-cudnn library path
python_version = f"python{sys.version_info.major}.{sys.version_info.minor}"
site_packages_path = f"/usr/local/lib/{python_version}/dist-packages/nvidia/cudnn/lib"

if os.path.exists(site_packages_path):
    os.environ["LD_LIBRARY_PATH"] = site_packages_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print(f"Set LD_LIBRARY_PATH to: {os.environ['LD_LIBRARY_PATH']}")
else:
    # If using virtual environments, check sys.prefix
    site_packages_path = f"{sys.prefix}/lib/{python_version}/site-packages/nvidia/cudnn/lib"
    if os.path.exists(site_packages_path):
        os.environ["LD_LIBRARY_PATH"] = site_packages_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        print(f"Set LD_LIBRARY_PATH to: {os.environ['LD_LIBRARY_PATH']}")
    else:
        print("Could not automatically locate pip's nvidia-cudnn path. Try Fix 2.")


Set LD_LIBRARY_PATH to: /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/lib64-nvidia


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
!cp -r gpu-output-deepseek2-16b /content/drive/MyDrive/
